# Interaction check on images

The same check as the audio interaction check, run on the Phase 1 image data. Entangling gates can only help a classifier if the label depends on interactions between inputs, not just on each input separately. This notebook measures whether the CLIP features of FaceForensics++ hold any such interaction signal.

At each input width it compares two families of classical models:

- **Additive models**, which score each input on its own and add the results: logistic regression on the raw values, logistic regression on values clipped to the training range, and the twin (logistic regression on the sine and cosine of each angle).
- **Interaction-capable models**: the twin plus every pairwise product of its sine and cosine features, which is what a circuit without entanglers can compute with ZZ readout, then histogram gradient boosting and a small MLP.

The Phase 1 fairness check already ran gradient boosting and an MLP against the twin on the best 8 and PLS 8 inputs and found no gap. This notebook adds 16 and 32 inputs and the full feature vector, uses the formal rule from the audio work, and runs a second pass with one manipulation held out.

The rule is the one in Amendment 2 of the audio plan. A gap opens at an input if the best interaction-capable model beats the best additive model by at least 0.01 AUC or 1 EER point, with a bootstrap interval that excludes 0.

- **Pass 1, all methods:** models train on the search-train frames and are scored on FF++ val, all four methods.
- **Pass 2, NeuralTextures held out:** models train on the search-train frames without NeuralTextures and are scored on val real videos against val NeuralTextures videos. This mirrors the unseen-attack setup of the audio check.

Attach the Phase 1 CLIP feature and feature prep datasets, and run on CPU. It should take 1 to 2 hours, most of it gradient boosting and the MLP on the full feature vector. Celeb-DF and FF++ test are not touched. Labels are 0 for real and 1 for fake.

## Setup

`encoder` picks the feature set. ViT-B/16 is the encoder behind all Phase 1 circuit results. ViT-L/14 needs its features and `prep_l14` datasets attached.

In [1]:
import os
import glob
import json
import time

import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve

seed = 0
encoder = "ViT-B/16"
c_grid = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
held_out = "NeuralTextures"
methods = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
out_dir = "/kaggle/working/interaction_check_images"
os.makedirs(out_dir, exist_ok=True)

feature_folder, prep_folder = {"ViT-B/16": ("features", "prep"), "ViT-L/14": ("features_l14", "prep_l14")}[encoder]

In [2]:
def find(pattern):
    for depth in range(8):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + pattern)
        if hits:
            return hits[0]
    raise FileNotFoundError(pattern)


prep_dir = os.path.dirname(find(f"{prep_folder}/ffpp_train_is_inner.npy"))
feature_dir = os.path.dirname(find(f"{feature_folder}/ffpp_train_clip.npy"))

meta, clip, pca_values, pca_angles = {}, {}, {}, {}
for split in ["train", "val"]:
    meta[split] = pd.read_csv(f"{prep_dir}/ffpp_{split}_rows.csv", dtype={"video_id": str})
    clip_rows = pd.read_csv(f"{feature_dir}/ffpp_{split}_rows.csv", dtype={"video_id": str})
    assert (clip_rows["path"].to_numpy() == meta[split]["path"].to_numpy()).all(), split
    x = np.load(f"{feature_dir}/ffpp_{split}_clip.npy").astype(np.float64)
    clip[split] = x / np.linalg.norm(x, axis=1, keepdims=True)
    pca_values[split] = np.load(f"{prep_dir}/ffpp_{split}_pca32.npy")
    pca_angles[split] = np.load(f"{prep_dir}/ffpp_{split}_angles32.npy")

params = np.load(f"{prep_dir}/pca32_params.npz")
is_inner = np.load(f"{prep_dir}/ffpp_train_is_inner.npy").astype(bool)
y_train = meta["train"]["label"].to_numpy()
y_val = meta["val"]["label"].to_numpy()
print(encoder, "features:", clip["train"].shape[1], " search-train frames:", (~is_inner).sum(),
      " val frames:", len(y_val))

ViT-B/16 features: 512  search-train frames: 39164  val frames: 22396


## Val videos and pairs

Every val frame belongs to a video, (method, video_id), and a video's score is the mean of its frame probabilities, as in Phase 1. Videos are also grouped by source pair: the fake 000_003 comes from originals 000 and 003, so both originals and all fakes made from them form one pair. The bootstrap resamples pairs, not single videos, because a fake and its source videos are not independent. That makes the intervals a little wider than the per-video bootstrap of Phase 1, and closer to the speaker bootstrap of the audio work.

In [3]:
def pair_of(video_id):
    a, b = video_id.split("_")
    return tuple(sorted([a, b]))


val = meta["val"]
real_pair = {}
for video_id in val.loc[val["label"] == 1, "video_id"].unique():
    for person in video_id.split("_"):
        real_pair[person] = pair_of(video_id)

frame_pair = [real_pair[v] if m == "original" else pair_of(v) for m, v in zip(val["method"], val["video_id"])]
video_of_frame = val.groupby(["method", "video_id"]).ngroup().to_numpy()
videos = pd.DataFrame({"video": video_of_frame, "y": y_val, "method": val["method"].to_numpy(),
                       "pair": [f"{a}_{b}" for a, b in frame_pair]}).groupby("video").first()
video_y = videos["y"].to_numpy()
video_method = videos["method"].to_numpy()
video_pair = videos["pair"].to_numpy()
print("val videos:", len(videos), " real:", (video_y == 0).sum(), " fake:", (video_y == 1).sum(),
      " pairs:", len(np.unique(video_pair)))


def video_scores(p_frames):
    return pd.Series(p_frames).groupby(video_of_frame).mean().to_numpy()

val videos: 700  real: 140  fake: 560  pairs: 70


## Feature maps

The product features follow from how a circuit without entanglers reads out. With one RY input per qubit, each qubit's Z expectation after a trainable rotation is a sin(a) + b cos(a) + c, and with ZZ readout on a product state the expectation of Z_i Z_j is the product of two such terms. So the pairwise model gets every product of a sine or cosine of input i with a sine or cosine of input j.

In [4]:
def sin_cos(a):
    return np.hstack([np.sin(a), np.cos(a)])


def pairwise_products(a):
    features = [np.sin(a), np.cos(a)]
    n = a.shape[1]
    for i in range(n):
        for j in range(i + 1, n):
            for f in [np.sin, np.cos]:
                for g in [np.sin, np.cos]:
                    features.append((f(a[:, i]) * g(a[:, j]))[:, None])
    return np.hstack(features)


def to_angles(z, low, high):
    return np.clip((z - low) / (high - low), 0.0, 1.0) * np.pi

## Models and scoring

Every model trains on frames and is scored on val videos. Logistic models choose C by val video AUC. Gradient boosting and the MLP each choose from 4 fixed settings the same way. Choosing on the same val videos that are reported makes every score slightly optimistic, and the same rule applies to every model, so the comparison stays fair.

In [5]:
def equal_error_rate(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    fnr = 1 - tpr
    i = np.argmin(np.abs(fnr - fpr))
    return 100 * (fpr[i] + fnr[i]) / 2


def best_by_val_auc(candidates, x_fit, y_fit, x_val, keep):
    best = None
    for setting, make_model in candidates:
        model = make_model()
        model.fit(x_fit, y_fit)
        p = video_scores(model.predict_proba(x_val)[:, 1])[keep]
        auc = roc_auc_score(video_y[keep], p)
        if best is None or auc > best["auc"]:
            best = {"setting": setting, "auc": auc, "p": p}
    return best


def logistic_candidates():
    return [(f"C={c}", lambda c=c: make_pipeline(
        StandardScaler(), LogisticRegression(C=c, class_weight="balanced", max_iter=5000))) for c in c_grid]


def boosting_candidates():
    settings = [(rate, leaves) for rate in [0.03, 0.1] for leaves in [15, 63]]
    return [(f"rate={rate}, leaves={leaves}", lambda rate=rate, leaves=leaves: HistGradientBoostingClassifier(
        learning_rate=rate, max_leaf_nodes=leaves, max_iter=500, class_weight="balanced", random_state=seed))
        for rate, leaves in settings]


def mlp_candidates():
    settings = [(hidden, alpha) for hidden in [(64,), (128, 64)] for alpha in [1e-4, 1e-2]]
    return [(f"hidden={hidden}, alpha={alpha}", lambda hidden=hidden, alpha=alpha: make_pipeline(
        StandardScaler(), MLPClassifier(hidden_layer_sizes=hidden, alpha=alpha, early_stopping=True,
                                        max_iter=300, random_state=seed))) for hidden, alpha in settings]

The paired pair bootstrap resamples the val source pairs with replacement and recomputes the difference between two models on the same resampled videos.

In [6]:
def paired_interval(y, pairs, p_additive, p_interaction, n_boot=1000):
    rng = np.random.default_rng(seed)
    pair_rows = {name: np.where(pairs == name)[0] for name in np.unique(pairs)}
    names = list(pair_rows)
    auc_gains, eer_drops = [], []
    for _ in range(n_boot):
        rows = np.concatenate([pair_rows[name] for name in rng.choice(names, len(names), replace=True)])
        if y[rows].min() == y[rows].max():
            continue
        auc_gains.append(roc_auc_score(y[rows], p_interaction[rows]) - roc_auc_score(y[rows], p_additive[rows]))
        eer_drops.append(equal_error_rate(y[rows], p_additive[rows]) - equal_error_rate(y[rows], p_interaction[rows]))
    return np.percentile(auc_gains, [2.5, 97.5]), np.percentile(eer_drops, [2.5, 97.5])

## One pass of the comparison

A pass fits PLS on its own training frames (normalized features, no scaling, as in Phase 1) with angle ranges from the 0.5th and 99.5th percentiles of those frames. PLS components are nested, so the first 8 and 16 columns of PLS 32 are PLS 8 and PLS 16. PCA and its angles are the Phase 1 ones, fitted on all train frames without labels.

For every input the pass fits all six models (three for the full feature vector, which has no angles), keeps the best additive and the best interaction-capable model, and measures the gap between them.

In [7]:
inputs = [("PCA", 8), ("PCA", 16), ("PCA", 32), ("PLS", 8), ("PLS", 16), ("PLS", 32), ("full", None)]


def run_pass(pass_name, fit_rows, keep):
    y_fit = y_train[fit_rows]
    y = video_y[keep]
    pairs = video_pair[keep]
    print(f"{pass_name}: {fit_rows.sum()} training frames, {keep.sum()} val videos")

    pls = PLSRegression(n_components=32, scale=False)
    pls.fit(clip["train"][fit_rows], y_fit)
    pls_fit = pls.transform(clip["train"][fit_rows])
    pls_val = pls.transform(clip["val"])
    pls_low = np.percentile(pls_fit, 0.5, axis=0)
    pls_high = np.percentile(pls_fit, 99.5, axis=0)

    reduced = {
        "PCA": {"z": (pca_values["train"][fit_rows], pca_values["val"]),
                "a": (pca_angles["train"][fit_rows], pca_angles["val"]),
                "range": (params["angle_lo"], params["angle_hi"])},
        "PLS": {"z": (pls_fit, pls_val),
                "a": (to_angles(pls_fit, pls_low, pls_high), to_angles(pls_val, pls_low, pls_high)),
                "range": (pls_low, pls_high)},
    }

    models, gaps = [], []
    for name, width in inputs:
        start = time.time()
        if width is None:
            z_fit, z_val = clip["train"][fit_rows], clip["val"]
            additive = {"logistic regression": (z_fit, z_val, logistic_candidates())}
            interaction = {"gradient boosting": (z_fit, z_val, boosting_candidates()),
                           "MLP": (z_fit, z_val, mlp_candidates())}
            label = f"full {clip['train'].shape[1]}"
        else:
            r = reduced[name]
            z_fit, z_val = r["z"][0][:, :width], r["z"][1][:, :width]
            a_fit, a_val = r["a"][0][:, :width], r["a"][1][:, :width]
            low, high = r["range"][0][:width], r["range"][1][:width]
            additive = {
                "logistic regression": (z_fit, z_val, logistic_candidates()),
                "logistic regression, clipped": (np.clip(z_fit, low, high), np.clip(z_val, low, high),
                                                 logistic_candidates()),
                "twin": (sin_cos(a_fit), sin_cos(a_val), logistic_candidates()),
            }
            interaction = {
                "twin + pairwise products": (pairwise_products(a_fit), pairwise_products(a_val),
                                             logistic_candidates()),
                "gradient boosting": (z_fit, z_val, boosting_candidates()),
                "MLP": (z_fit, z_val, mlp_candidates()),
            }
            label = f"{name} {width}"

        results = {}
        for family, group in [("additive", additive), ("interaction", interaction)]:
            for model_name, (x_fit, x_val, candidates) in group.items():
                best = best_by_val_auc(candidates, x_fit, y_fit, x_val, keep)
                results[(family, model_name)] = best
                models.append({"pass": pass_name, "input": label, "family": family, "model": model_name,
                               "setting": best["setting"], "auc": best["auc"], "eer %": equal_error_rate(y, best["p"])})

        best_additive = max((k for k in results if k[0] == "additive"), key=lambda k: results[k]["auc"])
        best_interaction = max((k for k in results if k[0] == "interaction"), key=lambda k: results[k]["auc"])
        p_add, p_int = results[best_additive]["p"], results[best_interaction]["p"]
        (gain_low, gain_high), (drop_low, drop_high) = paired_interval(y, pairs, p_add, p_int)

        auc_gain = roc_auc_score(y, p_int) - roc_auc_score(y, p_add)
        eer_drop = equal_error_rate(y, p_add) - equal_error_rate(y, p_int)
        opens = (auc_gain >= 0.01 and gain_low > 0) or (eer_drop >= 1.0 and drop_low > 0)

        row = {"pass": pass_name, "input": label, "best additive": best_additive[1],
               "best interaction": best_interaction[1], "auc additive": roc_auc_score(y, p_add),
               "auc gain": auc_gain, "auc gain 95% CI": f"[{gain_low:.4f}, {gain_high:.4f}]",
               "eer drop": eer_drop, "eer drop 95% CI": f"[{drop_low:.2f}, {drop_high:.2f}]", "gap opens": opens}
        for method in methods:
            sub = (video_method[keep] == "original") | (video_method[keep] == method)
            if (video_method[keep] == method).any():
                row[f"eer drop {method}"] = (equal_error_rate(y[sub], p_add[sub])
                                             - equal_error_rate(y[sub], p_int[sub]))
        gaps.append(row)
        print(f"  {label:9s} auc gain {auc_gain:+.4f}  eer drop {eer_drop:+.2f}  gap opens: {opens}  "
              f"({time.time() - start:.0f} s)")
    return models, gaps

## Run both passes

In [8]:
search_train = ~is_inner
train_method = meta["train"]["method"].to_numpy()

all_models, all_gaps = [], []
passes = [
    ("all methods", search_train, np.ones(len(video_y), dtype=bool)),
    (f"{held_out} held out", search_train & (train_method != held_out),
     (video_method == "original") | (video_method == held_out)),
]
for pass_name, fit_rows, keep in passes:
    models, gaps = run_pass(pass_name, fit_rows, keep)
    all_models += models
    all_gaps += gaps

all_models = pd.DataFrame(all_models)
gaps = pd.DataFrame(all_gaps)
all_models.to_csv(f"{out_dir}/all_models.csv", index=False)
gaps.to_csv(f"{out_dir}/gaps.csv", index=False)

all methods: 39164 training frames, 700 val videos
  PCA 8     auc gain +0.0142  eer drop +1.25  gap opens: False  (117 s)
  PCA 16    auc gain +0.0248  eer drop +1.34  gap opens: True  (157 s)
  PCA 32    auc gain +0.0135  eer drop +1.43  gap opens: False  (399 s)
  PLS 8     auc gain +0.0044  eer drop +0.62  gap opens: False  (79 s)
  PLS 16    auc gain -0.0001  eer drop -0.27  gap opens: False  (137 s)
  PLS 32    auc gain +0.0032  eer drop +0.27  gap opens: False  (312 s)
  full 512  auc gain -0.0303  eer drop -1.70  gap opens: False  (429 s)
NeuralTextures held out: 34268 training frames, 280 val videos
  PCA 8     auc gain +0.0256  eer drop +3.57  gap opens: True  (95 s)
  PCA 16    auc gain +0.0227  eer drop +1.79  gap opens: True  (160 s)
  PCA 32    auc gain +0.0022  eer drop -0.71  gap opens: False  (356 s)
  PLS 8     auc gain -0.0040  eer drop +2.86  gap opens: False  (83 s)
  PLS 16    auc gain -0.0041  eer drop +2.50  gap opens: False  (124 s)
  PLS 32    auc gain -0.0072

## Results

Every model at every input, then the gap table with its decision. The per-method columns in pass 1 compare each method's fakes against all real videos.

In [9]:
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
print(all_models.round({"auc": 4, "eer %": 2}).to_string(index=False))

                   pass    input      family                        model                        setting    auc  eer %
            all methods    PCA 8    additive          logistic regression                        C=0.001 0.6367  39.64
            all methods    PCA 8    additive logistic regression, clipped                          C=0.1 0.6363  39.55
            all methods    PCA 8    additive                         twin                         C=0.01 0.6471  39.64
            all methods    PCA 8 interaction     twin + pairwise products                        C=0.001 0.6567  38.75
            all methods    PCA 8 interaction            gradient boosting           rate=0.03, leaves=15 0.6479  37.77
            all methods    PCA 8 interaction                          MLP     hidden=(64,), alpha=0.0001 0.6613  38.39
            all methods   PCA 16    additive          logistic regression                        C=0.001 0.6898  36.16
            all methods   PCA 16    additive log

In [10]:
print(gaps.round(4).to_string(index=False))

decision = {"encoder": encoder, "rule": "Amendment 2 (audio plan)"}
for pass_name in gaps["pass"].unique():
    rows = gaps[gaps["pass"] == pass_name]
    opened_at = rows.loc[rows["gap opens"], "input"].tolist()
    decision[pass_name] = {"decision": "gap opens" if opened_at else "no gap", "inputs_with_gap": opened_at}
with open(f"{out_dir}/decision.json", "w") as f:
    json.dump(decision, f, indent=1)

print()
for pass_name in gaps["pass"].unique():
    print(f"{pass_name}: {decision[pass_name]['decision']}", decision[pass_name]["inputs_with_gap"] or "")

                   pass    input                best additive         best interaction  auc additive  auc gain    auc gain 95% CI  eer drop eer drop 95% CI  gap opens  eer drop Deepfakes  eer drop Face2Face  eer drop FaceSwap  eer drop NeuralTextures
            all methods    PCA 8                         twin                      MLP        0.6471    0.0142  [-0.0003, 0.0291]    1.2500   [-2.06, 5.18]      False              0.0000              1.7857             5.3571                   1.0714
            all methods   PCA 16                         twin twin + pairwise products        0.6999    0.0248   [0.0120, 0.0387]    1.3393   [-0.71, 4.82]       True              3.2143              2.8571             2.1429                   1.4286
            all methods   PCA 32                         twin twin + pairwise products        0.7622    0.0135  [-0.0036, 0.0285]    1.4286   [-1.88, 4.73]      False             -0.3571             -1.7857             3.2143                  -0.3